In [1]:
import numpy as np
import matplotlib
from matplotlib import pyplot as plot
from matplotlib.font_manager import FontProperties
from sklearn.linear_model import LinearRegression
from sklearn.decomposition import PCA
import sklearn.metrics
from sklearn.metrics import explained_variance_score, mean_squared_error, r2_score # https://scikit-learn.org/stable/auto_examples/linear_model/plot_ols.html
import statsmodels.api as sm
import pandas
import matplotlib.pyplot as plt
import scipy as sp
from scipy.odr import *
import random
import operator
import os

# For 3d plots. This import is necessary to have 3D plotting below
from mpl_toolkits.mplot3d import Axes3D

# For statistics. Requires statsmodels 5.0 or more
from statsmodels.formula.api import ols
from statsmodels.formula.api import wls
# Analysis of Variance (ANOVA) on linear models
from statsmodels.stats.anova import anova_lm

# Adapted from:
# https://www.datasciencecentral.com/profiles/blogs/linear-regression-in-python-use-of-numpy-scipy-and-statsmodels,
# https://realpython.com/linear-regression-in-python/#multiple-linear-regression

In [2]:
isochrone_data = np.transpose(np.genfromtxt('isochrones_deltalogt_is_0_005.dat',delimiter=' ',comments='#'))

In [3]:
# Indices for imported isochrone table. Magnitude indices can change depending on requested bands.
Zini_Index = 0 # Initial metallicity
MH_Index = 1 # ?
logAge_Index = 2 # log10 Age
Mini_Index = 3 # Initial Mass
int_IMF_Index = 4 # Integreated IMF
Mass_Index = 5 # Mass at current age
logL_Index = 6 # log10 Luminosity
logTe_Index = 7 # log10 Effective Temp.
logg_Index = 8 # log10 Surfrace gravity
label_Index = 9 # Label? Can be 0 or 1
McoreTP_Index = 10 # log10 Core mass?
C_O_Index = 11 # C/0
period0_Index = 12 # Fundamental mode period
period1_Index = 13 # 1st Overtone mode period
period2_Index = 14 # 2nd Overtone mode period
period3_Index = 15 # 3rd Overtone mode period
period4_Index = 16 # 4th Overtone mode period
pmode_Index = 17 # Pulsation mode (-1, 0, 1, 2, 3, or 4). -1 is not pulsating.
Mloss_Index = 18 # Current mass-loss rate
tau1m = 19 # optical depth?
X_Index = 20 # H composition
Y_Index = 21 # He composition
Xc_Index = 22 # ?
Xn_Index = 23 # ?
Xo_Index = 24 # ?
Cexcess_Index = 25 # Carbon excess
Z_Index = 26 # Metallicity
mbolmag_Index = 27 # Bolometric magnitude
Umag_Index = 28 # U band magnitude
Bmag_Index = 29 # B band magnitude
Vmag_Index = 30 # V band magnitude
Rmag_Index = 31 # R band magnitude
Imag_Index = 32 # I band magnitude
Jmag_Index = 33 # J band magnitude
Hmag_Index = 34 # H band magnitude
Kmag_Index = 35 # K band magnitude

In [4]:
np.shape(isochrone_data)

(36, 63970)

In [5]:
isochrone_data[logAge_Index]

array([ 9.     ,  9.     ,  9.     , ..., 10.12503, 10.12503, 10.12503])

In [6]:
# Restrict to only TP-AGB stars (presumably)
def cut_non_pulsators (isochrone_data):
    tp_iso_data = np.transpose(isochrone_data)
    rows_to_cut = []
    for i in range (0,len(tp_iso_data)):
        if tp_iso_data[i][pmode_Index] == -1:
            rows_to_cut.append(i)
    return np.transpose(np.delete(tp_iso_data,rows_to_cut,axis=0))

isochrone_data_only_pulsators = cut_non_pulsators (isochrone_data)

In [38]:
isochrone_data_only_pulsators[logAge_Index]

array([9.     , 9.     , 9.     , ..., 9.67502, 9.67502, 9.67502])

In [46]:
# Round Mass to nth decimal (max precision before numerical errors become readily apparent) and construct new array
n_round = 3
#np.round(isochrone_data_only_pulsators[Mini_Index],decimals=n_round)
rounded_isochrone_data = isochrone_data_only_pulsators
rounded_isochrone_data[Mini_Index,:] = np.round(isochrone_data_only_pulsators[Mini_Index],decimals=n_round)
print(isochrone_data_only_pulsators[Mini_Index])
print(rounded_isochrone_data[Mini_Index])
tp_rounded_data = np.transpose(rounded_isochrone_data)

Mini_sorted_ichrone_data = np.transpose(tp_rounded_data[tp_rounded_data[:, Mini_Index].argsort()])

[1.953 1.953 1.953 ... 1.097 1.097 1.097]
[1.953 1.953 1.953 ... 1.097 1.097 1.097]


In [48]:
Mini_sorted_ichrone_data[Mini_Index]

array([1.097, 1.097, 1.097, ..., 1.954, 1.954, 1.954])

In [49]:
Mini_sorted_ichrone_data[logAge_Index]

array([9.67502, 9.67502, 9.67502, ..., 9.     , 9.     , 9.     ])

In [50]:
# Cut array based on initial mass, then sort by age.
# Restrict cut arrays to stars that can be TP-AGB, at some point
def array_split_by_sorted_index (ichrone_data, index):
    # Preliminary Sort
    indices_of_same_index = []
    current_row = 0
    while current_row < np.shape(ichrone_data)[1]:
        if ichrone_data[Mini_Index][current_row] > 0.7: # Sub routine: Cut all the M-dwarf stars.
            indices = np.argwhere(ichrone_data[Mini_Index] == ichrone_data[Mini_Index][current_row]).flatten()
            
            indices_of_same_index.append(indices.tolist())
        else:
            indices = [current_row]
        current_row = np.nanmax(indices)+1
    #return indices_of_same_index
    # Now, return arrays of data split at indices
    arrays_of_equal_index = []
    for i in range(0, len(indices_of_same_index)):
        arrays_of_equal_index.append(np.transpose(ichrone_data)[indices_of_same_index[i]])
    return arrays_of_equal_index

array_Mini_split = array_split_by_sorted_index (Mini_sorted_ichrone_data,Mini_Index)

In [51]:
def cut_len_less_1 (sorted_and_split_ichrone_array):
    tp_iso_data = np.transpose(np.asarray(sorted_and_split_ichrone_array))
    rows_to_cut = []
    rows_to_keep = []
    for i in range (0,len(tp_iso_data)):
        if len(sorted_and_split_ichrone_array[i]) <= 1:
            rows_to_cut.append(i)
        else:
            rows_to_keep.append(i)
    print('Cut indices', rows_to_cut, 'for having length of 1 or smaller.')
    return np.transpose(np.delete(tp_iso_data,rows_to_cut,axis=0)).tolist()
    

In [52]:
array_Mini_split_trimmed = cut_len_less_1 (array_Mini_split)

Cut indices [102] for having length of 1 or smaller.


In [56]:
np.transpose(array_Mini_split[11])

array([[ 1.52000000e-03,  1.52000000e-03,  1.52000000e-03,
         1.52000000e-03,  1.52000000e-03,  1.52000000e-03,
         1.52000000e-03,  1.52000000e-03,  1.52000000e-03,
         1.52000000e-03,  1.52000000e-03,  1.52000000e-03,
         1.52000000e-03,  1.52000000e-03,  1.52000000e-03,
         1.52000000e-03,  1.52000000e-03,  1.52000000e-03,
         1.52000000e-03,  1.52000000e-03,  1.52000000e-03,
         1.52000000e-03,  1.52000000e-03],
       [-1.00761000e+00, -1.00761000e+00, -1.00761000e+00,
        -1.00761000e+00, -1.00761000e+00, -1.00761000e+00,
        -1.00761000e+00, -1.00761000e+00, -1.00761000e+00,
        -1.00761000e+00, -1.00761000e+00, -1.00761000e+00,
        -1.00761000e+00, -1.00761000e+00, -1.00761000e+00,
        -1.00761000e+00, -1.00761000e+00, -1.00761000e+00,
        -1.00761000e+00, -1.00761000e+00, -1.00761000e+00,
        -1.00761000e+00, -1.00761000e+00],
       [ 9.62001000e+00,  9.62001000e+00,  9.62001000e+00,
         9.62001000e+00,  9.6

In [54]:
# Find change in age over lnL. This is t_ev.
def calc_t_evs (sorted_ichrone_data):
    t_ev_list = []
    for i in range(0, len(sorted_ichrone_data)):
    #for i in range(0, 1):
        working_star = np.transpose(sorted_ichrone_data[i])
        #print(working_star)
        delta_t = 10**(np.nanmax(working_star[logAge_Index]))-10**(np.nanmin(working_star[logAge_Index]))
        #print(delta_t)
        delta_ln_L = np.log(10**(np.nanmax(working_star[logL_Index])))-np.log(10**(np.nanmin(working_star[logL_Index])))
        t_ev_of_i = delta_t/delta_ln_L
        t_ev_list.append(t_ev_of_i)
    return np.asarray(t_ev_list)

In [55]:
calc_t_evs (array_Mini_split_trimmed)

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])